In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
import random
from matplotlib import colors
import os, glob
from scipy.stats import pearsonr

In [ ]:
from matplotlib import font_manager
# Manually register the font file with matplotlib's fontManager.
font_manager.fontManager.addfont('../../data/Arial.ttf')

# Verify the font was registered.
print([f.name for f in font_manager.fontManager.ttflist if 'Arial' in f.name])
plt.rcParams['font.family'] = 'Arial'

In [ ]:
# Logarithmic model: R^2 = a + b * log(n).
def log_fit(x, a, b):
    return a + b * np.log(x)

In [ ]:
files1 = glob.glob('../../data/regression_outputs_new/Sampling/*/*/Fuse/Multi_Concat_pcahierachy/results.csv')
files2 = glob.glob('../../data/regression_outputs_new/Sampling/*/*/Fuse/Multi_Concat_pcahierachy_top1/results.csv')
files = files1 + files2
cities = []
for file in files:
    city = file.split('/')[-4]
    country = file.split('/')[-5]
    cities.append((city, country))
cities = list(set(cities))
len(cities)

In [ ]:
def load_data(city, country):
    if country in ['France', 'Portugal', 'Nigeria']:
        df = pd.read_csv(f'../../data/regression_outputs_new/Sampling/{country}/{city}/Fuse/Multi_Concat_pcahierachy_top1/results.csv')
    else:
        df = pd.read_csv(f'../../data/regression_outputs_new/Sampling/{country}/{city}/Fuse/Multi_Concat_pcahierachy/results.csv')
    label_sdg_file = f"../../data/processed/0labels/{country}.csv"
    labels_sdg = pd.read_csv(label_sdg_file)
    df = pd.merge(df, labels_sdg[['ID', 'SDG']], left_on='target', right_on='ID')
    baseline_df = pd.read_csv(f'../../data/regression_outputs_new/Ratio/{country}/{city}/Fuse/Multi_Concat/results.csv')
    baseline_df = pd.merge(baseline_df, labels_sdg[['ID', 'SDG']], left_on='target', right_on='ID')
    current_rows = []
    for target in df['target'].unique():
        data = df[df['target'] == target]
        x = data['ratio']
        y = data['R2']
        # Fit the R^2 curve.
        params_r2, covariance = curve_fit(log_fit, x, y)
        
        # Compute x at y = 0.8.
        x_08 = np.exp((0.8 - params_r2[0]) / params_r2[1]) 
        current_rows.append([target, data['SDG'].iloc[0], x_08])
    current_df = pd.DataFrame(current_rows, columns=['target', 'SDG', 'x_08'])
    current_df['type'] = 'current'
    baseline_rows = []
    for target in baseline_df['target'].unique():
        data = baseline_df[baseline_df['target'] == target]
        for fold in data['fold'].unique():
            data_fold = data[data['fold'] == fold]
            x = data_fold['ratio']
            y = data_fold['R2']
            # Fit the R^2 curve.
            params_r2, covariance = curve_fit(log_fit, x, y)
            
            # Compute x at y = 0.8.
            x_08 = np.exp((0.8 - params_r2[0]) / params_r2[1]) 
            baseline_rows.append([target, data['SDG'].iloc[0], x_08])
    baseline_res = pd.DataFrame(baseline_rows, columns=['target', 'SDG', 'x_08'])
    baseline_res['type'] = 'baseline'
    res = pd.concat([current_df, baseline_res])
    res['city'] = city
    res['country'] = country
    return res

In [ ]:
dfs = []
for city, country in cities:
    print(city, country)
    dfs.append(load_data(city, country))
data = pd.concat(dfs, ignore_index=True)
data

In [ ]:
data.sort_values(by=['country', 'city'], inplace=True)
data.reset_index(drop=True, inplace=True)
data

In [ ]:
city_data = []
for i, city in enumerate(data['city'].unique()):
    row = [city] + list(data[data['city'] == city][['x_08', 'type']].groupby('type').mean().values.flatten())
    city_data.append(row)
city_data = pd.DataFrame(city_data, columns=['city', 'baseline', 'current'])
city_data['diff'] = city_data['baseline'] - city_data['current']
city_data = city_data.merge(data[['city', 'country']].drop_duplicates(), on='city')
city_data

In [ ]:
city_data['current'].mean(), city_data['baseline'].mean(), city_data['diff'].mean()

In [ ]:
city_data.sort_values(by=['diff'], inplace=True)

In [ ]:
city_data['city'] = city_data['city'].apply(lambda x: x.replace('HongKong', 'Hong Kong'))
city_data['city'] = city_data['city'].apply(lambda x: x.replace('PortoAlegre', 'Porto Alegre'))
city_data['city'] = city_data['city'].apply(lambda x: x.replace('LosAngeles', 'Los Angeles'))
city_data['city'] = city_data['city'].apply(lambda x: x.replace('RiodeJaneiro', 'Rio de Janeiro'))
city_data['city'] = city_data['city'].apply(lambda x: x.replace('SanFrancisco', 'San Francisco'))
city_data['city'] = city_data['city'].apply(lambda x: x.replace('BeloHorizonte', 'Belo Horizonte'))
city_data

In [ ]:
# Discretize continuous values (equal-width binning).
def discretize_column(series, bins=10):
    # Equal-width binning.
    return pd.cut(series, bins=bins, labels=False, include_lowest=True)

# Entropy of a single column.
def calculate_entropy(series):
    value_counts = series.value_counts(normalize=True)
    entropy = -np.sum(value_counts * np.log2(value_counts))
    return entropy

# Entropy of the whole DataFrame.
def calculate_dataframe_entropy(df, bins=5):
    # Discretize each column.
    df_discretized = df.apply(discretize_column, bins=bins)
    entropies = df_discretized.apply(calculate_entropy)
    return entropies.mean()  # mean entropy across columns

In [ ]:
entropy_data = []
for i, city in enumerate(data['city'].unique()):
    df = data[data['city'] == city]
    current_mean = df[df['type'] == 'current']['x_08'].mean()
    baseline_mean = df[df['type'] == 'baseline']['x_08'].mean() 
    
    country = df['country'].iloc[0]
    feature_file = f"../../data/features/Unit/{country}/{city}/Fuse/Concat.pkl"
    features = pd.read_pickle(feature_file)
    entropy = calculate_dataframe_entropy(features[list(range(1536))])
    entropy_data.append([city, country, current_mean, baseline_mean, entropy])
entropy_df = pd.DataFrame(entropy_data, columns=['city', 'country', 'current_mean', 'baseline_mean', 'entropy'])
entropy_df

In [ ]:
entropy_df['current_ratio'] = entropy_df['current_mean'] *100
entropy_df['baseline_ratio'] = entropy_df['baseline_mean'] *100

In [ ]:
entropy_df['diff'] = entropy_df['baseline_mean'] - entropy_df['current_mean']

In [ ]:
entropy_df['diff_ratio'] = entropy_df['baseline_ratio'] - entropy_df['current_ratio']
entropy_df

In [ ]:
colors = ['#a0d3d5', '#f5deab', '#cbd9c0']

In [ ]:
import colorsys

In [ ]:
# Helper: darken a color (reduce lightness).
def darken_color(hex_color, factor=0.7):
    # HEX -> RGB.
    rgb = tuple(int(hex_color.lstrip('#')[i:i+2], 16) / 255 for i in (0, 2, 4))
    # Convert to HLS and lower lightness.
    h, l, s = colorsys.rgb_to_hls(*rgb)
    l = max(0, l * factor)  # smaller factor -> darker
    # Convert back to RGB then HEX.
    r, g, b = colorsys.hls_to_rgb(h, l, s)
    return f'#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}'

darken_colors = [darken_color(c) for c in colors]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

sns.regplot(data=entropy_df, x='entropy', y='current_ratio', ax=ax, label='Strategic sampling', ci=None, color=darken_colors[0], scatter_kws={'s': 100})
sns.regplot(data=entropy_df, x='entropy', y='baseline_ratio', ax=ax, label='Random sampling', ci=None, color=darken_colors[1], scatter_kws={'s': 100})

# Get x-axis data points.
x_vals = np.linspace(entropy_df['entropy'].min(), entropy_df['entropy'].max(), 100)

# Predicted values: current vs baseline.
current_model = np.polyfit(entropy_df['entropy'], entropy_df['current_ratio'], deg=1)
baseline_model = np.polyfit(entropy_df['entropy'], entropy_df['baseline_ratio'], deg=1)

# Compute predicted y.
current_y_vals = np.polyval(current_model, x_vals)
baseline_y_vals = np.polyval(baseline_model, x_vals)

# Assumes a 'city' column.
ny_row = entropy_df[entropy_df['city'] == 'NewYork'].iloc[0]
hk_row = entropy_df[entropy_df['city'] == 'HongKong'].iloc[0]

# Shade the area between the two curves.
ax.fill_between(x_vals, current_y_vals, baseline_y_vals, color=darken_colors[2], alpha=0.15)
# ax.fill_between(x_vals, current_y_vals, baseline_y_vals, color='blue', alpha=0.2, hatch='-')
# Keep only the bottom spine.
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(True)  # keep left spine
ax.spines['bottom'].set_visible(True)  # keep bottom spine
ax.spines['bottom'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)
# ax.spines['top'].set_linewidth(1)
# ax.spines['right'].set_linewidth(1)

ax.text(0.65, 0.03, f"Strategic sampling $R$: {pearsonr(entropy_df['entropy'], entropy_df['current_mean'])[0]:.2f}**",
        horizontalalignment='center',
        verticalalignment='center',
        transform=ax.transAxes,
        fontsize=24,
        color=darken_colors[0])
ax.text(0.65, 0.1, f"Random sampling $R$: {pearsonr(entropy_df['entropy'], entropy_df['baseline_mean'])[0]:.2f}***",
        horizontalalignment='center',
        verticalalignment='center',
        transform=ax.transAxes,
        fontsize=24,
        color=darken_colors[1])
# ax.text(0.7, 0.05, f"Difference $R$: {pearsonr(entropy_df['entropy'], entropy_df['diff'])[0]:.2f}**",
#         horizontalalignment='center',
#         verticalalignment='center',
#         transform=ax.transAxes,
#         fontsize=18,
#         color='gray')

print(pearsonr(entropy_df['entropy'], entropy_df['current_mean'])[1], pearsonr(entropy_df['entropy'], entropy_df['baseline_mean'])[1], pearsonr(entropy_df['entropy'], entropy_df['diff'])[1])

ax.tick_params(axis='x', labelsize=22)  # x-tick label size
ax.tick_params(axis='y', labelsize=22)  # y-tick label size

plt.ylabel('Average sampling ratio (%)', fontsize=24)
plt.xlabel('Entropy', fontsize=24)
# plt.legend(frameon=False, fontsize=16)
# close legend
plt.legend().set_visible(False)  # hide legend
plt.savefig('../../data/figure_assets/entropy_two.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
sns.regplot(data=entropy_df, x='entropy', y='diff_ratio', ax=ax, color=darken_colors[2], scatter_kws={'s': 100})
# plt.text(0.1, 0.9, f"Pearson Correlation: {pearsonr(entropy_df['entropy'], entropy_df['diff'])[0]:.4f}\n$p$={pearsonr(entropy_df['entropy'], entropy_df['diff'])[1]:4f}", fontsize=14, transform=ax.transAxes)
plt.text(0.05, 0.9, f"$R$: {pearsonr(entropy_df['entropy'], entropy_df['diff'])[0]:.2f}**", fontsize=24, transform=ax.transAxes, color=darken_colors[2])
ax.set_xlabel('Entropy', fontsize=24)
ax.set_ylabel('Difference of sampling ratio (%)', fontsize=24)

# Keep only the bottom spine.
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(True)  # keep left spine
ax.spines['bottom'].set_visible(True) # keep bottom spine
ax.spines['bottom'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)
# ax.spines['top'].set_linewidth()
# ax.spines['right'].set_linewidth(1)
# 
y_min, y_max = ax.get_ylim()  # current y-axis range
ax.set_yticks(np.arange(-3, 13, 5))

ax.tick_params(axis='both', which='major', labelsize=22)
plt.savefig('../../data/figure_assets/entropy_diff.svg', bbox_inches='tight')
plt.show()